# RBC instance segmentation — yolo11n-seg training

Fine-tunes `yolo11n-seg.pt` on `yolo_seg_dataset` (4,677 images, Cellpose-bootstrapped contour labels, WBC-cleaned, mistakes/bad-region pruned).

Output: `runs/rbc_seg_v1/weights/best.pt` — loads and predicts contours directly, no classical-CV post-processing needed.

In [1]:
import torch
from ultralytics import YOLO

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

CUDA available: True
GPU: NVIDIA GeForce RTX 3050


## Config

In [2]:
from pathlib import Path

DATASET_YAML = Path(r"F:\Livo\Data - 2026\Rbc\yolo_seg_dataset\dataset.yaml")
RUNS_DIR = Path(r"F:\Livo\Data - 2026\Rbc\runs")
RUN_NAME = "rbc_seg_maskratio_imgz1024"  

EPOCHS = 15
IMG_SIZE = 1024
BATCH = 1  
PATIENCE = 0

print(DATASET_YAML.read_text())

path: F:/Livo/Data - 2026/Rbc/yolo_seg_dataset
train: images/train
val: images/val
nc: 1
names:
  0: rbc



## Sanity-check the dataset before spending GPU time on it

In [3]:
for split in ["train", "val"]:
    img_dir = DATASET_YAML.parent / "images" / split
    lbl_dir = DATASET_YAML.parent / "labels" / split
    n_img = len(list(img_dir.glob("*.jpg")))
    n_lbl = len(list(lbl_dir.glob("*.txt")))
    print(f"{split}: {n_img} images, {n_lbl} labels")

train: 4177 images, 4177 labels
val: 500 images, 500 labels


## Train

Loads pretrained COCO weights and fine-tunes on the RBC dataset — not training from scratch, so it already knows general edges/shapes going in.

In [5]:
model = YOLO("yolo11n-seg.pt")

results = model.train(
    data=str(DATASET_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,
    patience=PATIENCE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    workers=0,      # Windows + Jupyter: multiprocessing DataLoader workers deadlock at epoch start, this avoids it
    plots=False,    # label/results plotting chokes on ~712k instances (170/img avg) -- big RAM spike, skip it
    mosaic=0.0,     # mosaic stitches 4 full images together before cropping -- momentary 4x memory, disable on 8GB
    mask_ratio = 4,)

New https://pypi.org/project/ultralytics/8.4.135 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.67  Python-3.11.15 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=F:\Livo\Data - 2026\Rbc\yolo_seg_dataset\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937

## Validate the best checkpoint

In [8]:
best_weights = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
best_model = YOLO(str(best_weights))
metrics = best_model.val(data=str(DATASET_YAML))

print("mask mAP50-95:", metrics.seg.map)
print("mask mAP50:   ", metrics.seg.map50)

FileNotFoundError: [Errno 2] No such file or directory: 'F:\\Livo\\Data - 2026\\Rbc\\runs\\rbc_seg_v1\\weights\\best.pt'

## Quick visual check — run inference on one val image and draw the predicted contours

In [ ]:
import cv2
import numpy as np

val_dir = DATASET_YAML.parent / "images" / "val"
sample_path = sorted(val_dir.glob("*.jpg"))[0]

result = best_model.predict(str(sample_path), conf=0.25)[0]

img = cv2.imread(str(sample_path))
for contour in result.masks.xy:
    pts = contour.astype(np.int32).reshape(-1, 1, 2)
    cv2.polylines(img, [pts], True, (0, 255, 0), 1, cv2.LINE_AA)

out_path = DATASET_YAML.parent / "sample_prediction.jpg"
cv2.imwrite(str(out_path), img)
print(f"{len(result.masks.xy)} cells detected -> {out_path}")

## Next steps

- `runs/rbc_seg_v1/weights/best.pt` is the file to load for real inference (`YOLO("best.pt")`).
- `runs/rbc_seg_v1/` also has training curves, confusion matrix, and prediction-vs-label mosaics from `ultralytics` — worth a look before trusting the model.
- Batch inference over a folder + write results to parquet: see the earlier `contour_x`/`contour_y` schema discussion — same `result.masks.xy` output, looped over every file.